In [1]:
from datasets import load_dataset
import numpy as np
import evaluate
from peft import get_peft_model, LoraConfig, TaskType, PeftModel, PeftConfig
from transformers import XLMRobertaTokenizer, XLMRobertaForSequenceClassification, Trainer, TrainingArguments

c:\Users\lewy7\Documents\GitHub\roberta-hypernet-custom\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

# #* Continue finetuning using peft from checkpoint

# checkpoint_path = "./results_lora/checkpoint-1800" # "xlm-roberta-base"
# tokenizer = XLMRobertaTokenizer.from_pretrained(checkpoint_path)

# peft_config = PeftConfig.from_pretrained(checkpoint_path)

# base_model = XLMRobertaForSequenceClassification.from_pretrained(peft_config.base_model_name_or_path, num_labels=2)
# model = PeftModel.from_pretrained(base_model, checkpoint_path)

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:

#* Init finetuning using peft 

# Load tokenizer and model
model_name = "./results_lora/checkpoint-1800" # "xlm-roberta-base"
tokenizer = XLMRobertaTokenizer.from_pretrained(model_name)
base_model = XLMRobertaForSequenceClassification.from_pretrained(model_name, num_labels=2)

# peft_config = LoraConfig(
#     task_type=TaskType.SEQ_CLS,
#     inference_mode=False,
#     r=32,
#     lora_alpha=16,
#     lora_dropout=0.05
# )
# model = get_peft_model(base_model, peft_config)

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
# Total number of parameters
total_params = sum(p.numel() for p in base_model.parameters())

# Trainable parameters (typically only a subset with LoRA)
trainable_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters: 279,816,964
Trainable parameters: 0


In [ ]:
# Load CoLA dataset
dataset = load_dataset("glue", "cola")

In [ ]:
# Tokenize
def tokenize_function(examples):
    return tokenizer(examples["sentence"], truncation=True, padding="max_length")

encoded_dataset = dataset.map(tokenize_function, batched=True)
encoded_dataset = encoded_dataset.rename_column("label", "labels")
encoded_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
metric = evaluate.load("glue", "cola")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results_lora",
    eval_strategy="steps",
    eval_steps=300,
    save_strategy="steps",
    save_steps=600,
    learning_rate=5e-4,
    per_device_train_batch_size=20, #* maybe think about adding gradient accumulation
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs_lora",
    load_best_model_at_end=True,
    metric_for_best_model="matthews_correlation",
    dataloader_num_workers=4
)

In [ ]:
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

C:\Users\lewy7\AppData\Local\Temp\ipykernel_12576\599301337.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
trainer.train()